In [1]:
# System
import os
import sys

os.environ["KERAS_BACKEND"] = "jax"
sys.path.append("../..")

In [2]:
# Setup
import json
from importlib import import_module
from pathlib import Path

import numpy as np
from keras import ops
from rich.table import Table

from src.models import GradientBoostedDecisionTree as BDT
from src.models import LearnableCutFlowParallel as LCF_PAR
from src.models import LearnableCutFlowSequential as LCF_SEQ
from src.models import MultiLayerPerceptron as MLP
from src.utils import Timer, load_model, print, to_numpy

In [3]:
# Parameters
rerun = False
n_runs = 10

# Dataset
dataset = "real1"  # *
selected_feature_indices = [0, 2, 4, 5]  # *
n_samples = 200000
seed = 42


# Model
centers = [80, 0.15, 0.025, 2, 2, 0.3]  # *
features = [
    r"$M_{jet}$",
    r"$C_2^{\beta=1}$",
    r"$C_2^{\beta=2}$",
    r"$D_2^{\beta=1}$",
    r"$D_2^{\beta=2}$",
    r"$\tau_{21}^{\beta=1}$",
]  # *

n_epochs = 200
batch_size = 512
verbose = 0

if not rerun and Path("results.json").exists():
    with open("results.json", "r") as f:
        results = json.load(f)
else:
    results = {}

In [4]:
# Dataset *
module = import_module(f"src.datasets.{dataset}")
load_data = getattr(module, "load_data")
(x_train, y_train), (x_test, y_test) = load_data(n_samples, seed)

x_train = x_train[:, selected_feature_indices]
x_test = x_test[:, selected_feature_indices]
centers = [centers[i] for i in selected_feature_indices]
features = [features[i] for i in selected_feature_indices]

selection = np.ones_like(x_train[:, 0], dtype=bool)
for i in range(x_train.shape[1]):
    p05 = np.percentile(x_train[:, i], 5)
    p95 = np.percentile(x_train[:, i], 95)
    selection = (p05 < x_train[:, i]) & (x_train[:, i] < p95) & selection

x_train = x_train[selection]
y_train = y_train[selection]

selection = np.ones_like(x_test[:, 0], dtype=bool)
for i in range(x_test.shape[1]):
    p05 = np.percentile(x_test[:, i], 5)
    p95 = np.percentile(x_test[:, i], 95)
    selection = (p05 < x_test[:, i]) & (x_test[:, i] < p95) & selection

x_test = x_test[selection]
y_test = y_test[selection]

print(f"{x_train.shape=}")
print(f"{y_train.shape=}")
print(f"{x_test.shape=}")
print(f"{y_test.shape=}")

x_train.shape=(76993, 4)
y_train.shape=(76993, 1)
x_test.shape=(76874, 4)
y_test.shape=(76874, 1)


In [5]:
# Model: BDT
bdt_name = "bdt"
bdt_ckpt_paths = [Path(f"checkpoints/{bdt_name}@{i + 1}.pkl") for i in range(n_runs)]

bdt_ckpts = []
if rerun or not all(ckpt.exists() for ckpt in bdt_ckpt_paths):
    records = []
    for i in range(n_runs):
        print(f"Processing {bdt_name}@{i + 1}...")

        bdt = BDT(input_shape=x_train.shape, name=bdt_name)
        bdt.compile(optimizer="adam", loss="crossentropy")

        with Timer() as timer:
            bdt.fit(
                x_train,
                y_train.squeeze(),
                batch_size=batch_size,
                epochs=n_epochs,
                verbose=verbose,
            )
        records.append(timer.record)

        bdt.save(bdt_ckpt_paths[i])
        bdt_ckpts.append(load_model(bdt_ckpt_paths[i]))

    results[bdt_name] = {
        "training_time_mean": np.mean(records),
        "training_time_std": np.std(records),
    }

print(
    "Training time: "
    f"{results[bdt_name]['training_time_mean']:.2f} ± "
    f"{results[bdt_name]['training_time_std']:.2f} seconds"
)

KeyError: 'training_time_mean'

In [6]:
# Model: MLP
mlp_name = "mlp"
mlp_ckpt_paths = [Path(f"checkpoints/{mlp_name}@{i + 1}.keras") for i in range(n_runs)]

mlp_ckpts = []
if rerun or not all(ckpt.exists() for ckpt in mlp_ckpt_paths):
    records = []
    for i in range(n_runs):
        print(f"Processing {mlp_name}@{i + 1}...")

        mlp = MLP(x_train.shape, name=mlp_name)
        mlp.adapt(x_train)
        mlp.compile(optimizer="adam", loss="crossentropy")

        with Timer() as timer:
            mlp.fit(
                x_train,
                y_train,
                batch_size=batch_size,
                epochs=n_epochs,
                verbose=verbose,
            )
        records.append(timer.record)

        mlp.save(mlp_ckpt_paths[i])
        mlp_ckpts.append(load_model(mlp_ckpt_paths[i]))

    results[mlp_name] = {
        "training_time_mean": np.mean(records),
        "training_time_std": np.std(records),
    }

print(
    "Training time: "
    f"{results[mlp_name]['training_time_mean']:.2f} ± "
    f"{results[mlp_name]['training_time_std']:.2f} seconds"
)

Processing mlp@1...
Epoch 1/200
151/151 - 11s - 74ms/step - loss: 0.4085
Epoch 2/200
151/151 - 1s - 8ms/step - loss: 0.3364
Epoch 3/200
151/151 - 0s - 1ms/step - loss: 0.3310
Epoch 4/200
151/151 - 0s - 1ms/step - loss: 0.3280
Epoch 5/200
151/151 - 0s - 1ms/step - loss: 0.3259
Epoch 6/200
151/151 - 0s - 2ms/step - loss: 0.3248
Epoch 7/200
151/151 - 0s - 1ms/step - loss: 0.3240
Epoch 8/200
151/151 - 0s - 1ms/step - loss: 0.3236
Epoch 9/200
151/151 - 0s - 1ms/step - loss: 0.3230
Epoch 10/200
151/151 - 0s - 1ms/step - loss: 0.3231
Epoch 11/200
151/151 - 0s - 1ms/step - loss: 0.3226
Epoch 12/200
151/151 - 0s - 1ms/step - loss: 0.3224
Epoch 13/200
151/151 - 0s - 1ms/step - loss: 0.3217
Epoch 14/200
151/151 - 0s - 2ms/step - loss: 0.3216
Epoch 15/200
151/151 - 0s - 3ms/step - loss: 0.3213
Epoch 16/200
151/151 - 0s - 3ms/step - loss: 0.3212
Epoch 17/200
151/151 - 0s - 3ms/step - loss: 0.3212
Epoch 18/200
151/151 - 0s - 1ms/step - loss: 0.3205
Epoch 19/200
151/151 - 0s - 1ms/step - loss: 0.3204

In [7]:
# Model: LCF(parallel)
lcf_par_name = "lcf_par"
lcf_par_ckpt_paths = [
    Path(f"checkpoints/{lcf_par_name}@{i + 1}.keras") for i in range(n_runs)
]

lcf_par_ckpts = []
if rerun or not all(ckpt.exists() for ckpt in lcf_par_ckpt_paths):
    records = []
    for i in range(n_runs):
        print(f"Processing {lcf_par_name}@{i + 1}...")

        lcf_par = LCF_PAR(x_train.shape, centers, features=features, name=lcf_par_name)
        lcf_par.adapt(x_train)
        lcf_par.compile(optimizer="adam", loss="crossentropy")

        with Timer() as timer:
            lcf_par.fit(
                x_train,
                y_train,
                batch_size=batch_size,
                epochs=n_epochs,
                verbose=verbose,
            )
        records.append(timer.record)

        lcf_par.save(lcf_par_ckpt_paths[i])
        lcf_par_ckpts.append(load_model(lcf_par_ckpt_paths[i]))

    results[lcf_par_name] = {
        "training_time_mean": np.mean(records),
        "training_time_std": np.std(records),
    }

print(
    "Training time: "
    f"{results[lcf_par_name]['training_time_mean']:.2f} ± "
    f"{results[lcf_par_name]['training_time_std']:.2f} seconds"
)

Processing lcf_par@1...
Epoch 1/200
151/151 - 5s - 33ms/step - loss: 0.3461
Epoch 2/200
151/151 - 1s - 10ms/step - loss: 0.3331
Epoch 3/200
151/151 - 0s - 2ms/step - loss: 0.3240
Epoch 4/200
151/151 - 0s - 1ms/step - loss: 0.3177
Epoch 5/200
151/151 - 0s - 1ms/step - loss: 0.3134
Epoch 6/200
151/151 - 0s - 1ms/step - loss: 0.3102
Epoch 7/200
151/151 - 0s - 2ms/step - loss: 0.3076
Epoch 8/200
151/151 - 0s - 2ms/step - loss: 0.3049
Epoch 9/200
151/151 - 0s - 2ms/step - loss: 0.3021
Epoch 10/200
151/151 - 0s - 2ms/step - loss: 0.2991
Epoch 11/200
151/151 - 0s - 2ms/step - loss: 0.2963
Epoch 12/200
151/151 - 0s - 2ms/step - loss: 0.2939
Epoch 13/200
151/151 - 0s - 2ms/step - loss: 0.2919
Epoch 14/200
151/151 - 0s - 2ms/step - loss: 0.2902
Epoch 15/200
151/151 - 0s - 2ms/step - loss: 0.2888
Epoch 16/200
151/151 - 0s - 2ms/step - loss: 0.2877
Epoch 17/200
151/151 - 0s - 2ms/step - loss: 0.2867
Epoch 18/200
151/151 - 0s - 2ms/step - loss: 0.2859
Epoch 19/200
151/151 - 0s - 2ms/step - loss: 0.

In [8]:
# Model: LCF(sequential)
lcf_seq_name = "lcf_seq"
lcf_seq_ckpt_paths = [
    Path(f"checkpoints/{lcf_seq_name}@{i + 1}.keras") for i in range(n_runs)
]

lcf_seq_ckpts = []
if rerun or not all(ckpt.exists() for ckpt in lcf_seq_ckpt_paths):
    records = []
    for i in range(n_runs):
        print(f"Processing {lcf_seq_name}@{i + 1}...")

        lcf_seq = LCF_SEQ(x_train.shape, centers, features=features, name=lcf_seq_name)
        lcf_seq.adapt(x_train)
        lcf_seq.compile(optimizer="adam", loss="crossentropy")

        with Timer() as timer:
            lcf_seq.fit(
                x_train,
                y_train,
                batch_size=batch_size,
                epochs=n_epochs,
                verbose=verbose,
            )
        records.append(timer.record)

        lcf_seq.save(lcf_seq_ckpt_paths[i])
        lcf_seq_ckpts.append(load_model(lcf_seq_ckpt_paths[i]))

    results[lcf_seq_name] = {
        "training_time_mean": np.mean(records),
        "training_time_std": np.std(records),
    }

print(
    "Training time: "
    f"{results[lcf_seq_name]['training_time_mean']:.2f} ± "
    f"{results[lcf_seq_name]['training_time_std']:.2f} seconds"
)

Processing lcf_seq@1...
Epoch 1/200
151/151 - 5s - 36ms/step - loss: 0.0946
Epoch 2/200
151/151 - 2s - 11ms/step - loss: 0.0901
Epoch 3/200
151/151 - 0s - 1ms/step - loss: 0.0867
Epoch 4/200
151/151 - 0s - 2ms/step - loss: 0.0841
Epoch 5/200
151/151 - 0s - 2ms/step - loss: 0.0821
Epoch 6/200
151/151 - 0s - 1ms/step - loss: 0.0805
Epoch 7/200
151/151 - 0s - 2ms/step - loss: 0.0789
Epoch 8/200
151/151 - 0s - 1ms/step - loss: 0.0786
Epoch 9/200
151/151 - 0s - 2ms/step - loss: 0.1041
Epoch 10/200
151/151 - 0s - 1ms/step - loss: 0.1449
Epoch 11/200
151/151 - 0s - 2ms/step - loss: 0.1518
Epoch 12/200
151/151 - 0s - 1ms/step - loss: 0.1551
Epoch 13/200
151/151 - 0s - 2ms/step - loss: 0.1576
Epoch 14/200
151/151 - 0s - 2ms/step - loss: 0.1597
Epoch 15/200
151/151 - 0s - 2ms/step - loss: 0.1610
Epoch 16/200
151/151 - 0s - 2ms/step - loss: 0.1613
Epoch 17/200
151/151 - 0s - 2ms/step - loss: 0.1606
Epoch 18/200
151/151 - 0s - 2ms/step - loss: 0.1602
Epoch 19/200
151/151 - 0s - 1ms/step - loss: 0.

In [10]:
# Analysis: metrics
rerun = True
y_true = y_test

table = Table(title="Model Performance Comparison")
table.add_column("#", justify="center", style="cyan", no_wrap=True)
table.add_column("Model", style="magenta")
table.add_column("TP", justify="right", style="green")
table.add_column("FP", justify="right", style="red")
table.add_column("Accuracy", justify="right", style="blue")
table.add_column("Precision", justify="right", style="blue")
table.add_column("Significance", justify="right", style="yellow")
table.add_column("Time(s)", justify="right", style="yellow")

if rerun or not Path("resultsx10.json").exists():
    for ckpts in [bdt_ckpts, mlp_ckpts, lcf_par_ckpts, lcf_seq_ckpts]:
        print(f"Processing {ckpts[0].name}...")

        tp_list = []
        fp_list = []
        accuracy_list = []
        precision_list = []
        significance_list = []

        for i, ckpt in enumerate(ckpts):
            y_pred = ckpt.predict(x_test, batch_size=batch_size, verbose=0)
            y_pred = ops.all(y_pred > 0.5, axis=1, keepdims=True)

            tp = to_numpy(ops.sum((y_true == 1) & (y_pred == 1)))
            fp = to_numpy(ops.sum((y_true == 0) & (y_pred == 1)))
            tn = to_numpy(ops.sum((y_true == 0) & (y_pred == 0)))
            fn = to_numpy(ops.sum((y_true == 1) & (y_pred == 0)))

            accuracy = (tp + tn) / (tp + tn + fp + fn)
            precision = tp / (tp + fp)

            s = tp / (tp + tn + fp + fn) * 3000 * 1000 * 0.7644
            b = fp / (tp + tn + fp + fn) * 3000 * 1000 * 1.806 * 1e5
            significance = s / np.sqrt(b)

            tp_list.append(tp)
            fp_list.append(fp)
            accuracy_list.append(accuracy)
            precision_list.append(precision)
            significance_list.append(significance)

        tp_mean = np.mean(tp_list)
        fp_mean = np.mean(fp_list)
        accuracy_mean = np.mean(accuracy_list)
        precision_mean = np.mean(precision_list)
        significance_mean = np.mean(significance_list)

        tp_std = np.std(tp_list)
        fp_std = np.std(fp_list)
        accuracy_std = np.std(accuracy_list)
        precision_std = np.std(precision_list)
        significance_std = np.std(significance_list)

        results[ckpts[0].name].update(
            {
                "tp_mean": tp_mean.tolist(),
                "tp_std": tp_std.tolist(),
                "fp_mean": fp_mean.tolist(),
                "fp_std": fp_std.tolist(),
                "accuracy_mean": accuracy_mean.tolist(),
                "accuracy_std": accuracy_std.tolist(),
                "precision_mean": precision_mean.tolist(),
                "precision_std": precision_std.tolist(),
                "significance_mean": significance_mean.tolist(),
                "significance_std": significance_std.tolist(),
            }
        )

        with open("resultsx10.json", "w") as f:
            json.dump(results, f, indent=4)

for i, (name, metrics) in enumerate(results.items()):
    tp_mean = metrics["tp_mean"]
    tp_std = metrics["tp_std"]
    fp_mean = metrics["fp_mean"]
    fp_std = metrics["fp_std"]
    accuracy_mean = metrics["accuracy_mean"]
    accuracy_std = metrics["accuracy_std"]
    precision_mean = metrics["precision_mean"]
    precision_std = metrics["precision_std"]
    significance_mean = metrics["significance_mean"]
    significance_std = metrics["significance_std"]
    training_time_mean = metrics["training_time_mean"]
    training_time_std = metrics["training_time_std"]

    table.add_row(
        str(i + 1),
        name,
        f"{tp_mean:.0f}\n± {tp_std:.0f}",
        f"{fp_mean:.0f}\n± {fp_std:.0f}",
        f"{accuracy_mean:.4f}\n± {accuracy_std:.4f}",
        f"{precision_mean:.4f}\n± {precision_std:.4f}",
        f"{significance_mean:.4f}\n± {significance_std:.4f}",
        f"{training_time_mean:.2f}\n± {training_time_std:.2f}",
    )

print(table)

Processing bdt...


Processing mlp...
Processing lcf_par...
Processing lcf_seq...
                         Model Performance Comparison                          
┏━━━┳━━━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━┓
┃ # ┃ Model   ┃    TP ┃    FP ┃ Accuracy ┃ Precision ┃ Significance ┃ Time(s) ┃
┡━━━╇━━━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━┩
│ 1 │ bdt     │ 41006 │  6709 │   0.8657 │    0.8594 │       5.6254 │   16.22 │
│   │         │   ± 0 │   ± 0 │ ± 0.0000 │  ± 0.0000 │     ± 0.0000 │  ± 1.45 │
│ 2 │ mlp     │ 40834 │  6424 │   0.8671 │    0.8641 │       5.7259 │   56.49 │
│   │         │ ± 177 │ ± 165 │ ± 0.0004 │  ± 0.0025 │     ± 0.0495 │  ± 3.93 │
│ 3 │ lcf_par │ 27665 │  3679 │   0.7315 │    0.8826 │       5.1248 │   54.37 │
│   │         │ ± 114 │  ± 11 │ ± 0.0014 │  ± 0.0004 │     ± 0.0195 │  ± 1.74 │
│ 4 │ lcf_seq │ 40346 │  7027 │   0.8530 │    0.8517 │       5.4099 │   57.39 │
│   │         │ ± 350 │ ± 238 │ ± 0.0015 │  ± 0.0032 │    